In [44]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    accuracy_score, confusion_matrix, classification_report
)
import xgboost as xgb
import lightgbm as lgb

X_train = pd.read_csv('../data/processed/X_train.csv')          # SMOTE-balanced
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
X_train_orig = pd.read_csv('../data/processed/X_train_original.csv')  # imbalanced
y_train_orig = pd.read_csv('../data/processed/y_train_original.csv').squeeze()
X_test = pd.read_csv('../data/processed/X_test.csv')
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()
import mlflow

# Force MLflow to log straight to your active database file
mlflow.set_tracking_uri("sqlite:///D:/Customer Churn Prediction/Customer-Churn-Prediction/notebooks/mlflow.db")
mlflow.set_experiment("churn-prediction")
print("Active MLflow Database Path:", mlflow.get_tracking_uri())

2026/06/17 20:44:00 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/17 20:44:00 INFO mlflow.store.db.utils: Updating database tables
2026/06/17 20:44:02 INFO mlflow.tracking.fluent: Experiment with name 'churn-prediction' does not exist. Creating a new experiment.


Active MLflow Database Path: sqlite:///D:/Customer Churn Prediction/Customer-Churn-Prediction/notebooks/mlflow.db


In [45]:
def evaluate_model(model, X_test, y_test, model_name="model"):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba)
    }

    print(f"\n{'='*40}\n{model_name}\n{'='*40}")
    for k, v in metrics.items():
        print(f"{k:>10}: {v:.4f}")
    print("\n" + classification_report(y_test, y_pred, target_names=['Retained', 'Churned']))

    return metrics, y_pred, y_proba

In [46]:
with mlflow.start_run(run_name="logistic_regression_baseline"):
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    metrics, y_pred, y_proba = evaluate_model(model, X_test, y_test, "Logistic Regression")

    mlflow.log_params({"model_type": "LogisticRegression", "max_iter": 1000})
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(model, "model")

d:\Customer Churn Prediction\Customer-Churn-Prediction\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2026/06/17 20:44:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/17 20:44:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The r


Logistic Regression
  accuracy: 0.7800
 precision: 0.5755
    recall: 0.6524
        f1: 0.6115
   roc_auc: 0.8332

              precision    recall  f1-score   support

    Retained       0.87      0.83      0.85      1035
     Churned       0.58      0.65      0.61       374

    accuracy                           0.78      1409
   macro avg       0.72      0.74      0.73      1409
weighted avg       0.79      0.78      0.78      1409



In [47]:
with mlflow.start_run(run_name="random_forest"):
    params = {"n_estimators": 200, "max_depth": 10, "random_state": 42, "n_jobs": -1}
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)

    metrics, y_pred, y_proba = evaluate_model(model, X_test, y_test, "Random Forest")

    mlflow.log_params(params)
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(model, "model")

2026/06/17 20:44:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/17 20:44:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Random Forest
  accuracy: 0.7700
 precision: 0.5551
    recall: 0.6738
        f1: 0.6087
   roc_auc: 0.8337

              precision    recall  f1-score   support

    Retained       0.87      0.80      0.84      1035
     Churned       0.56      0.67      0.61       374

    accuracy                           0.77      1409
   macro avg       0.71      0.74      0.72      1409
weighted avg       0.79      0.77      0.78      1409



In [48]:
with mlflow.start_run(run_name="xgboost_smote"):
    params = {
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 6,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "random_state": 42,
        "eval_metric": "logloss"
    }
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train)

    metrics, y_pred, y_proba = evaluate_model(model, X_test, y_test, "XGBoost (SMOTE)")

    mlflow.log_params(params)
    mlflow.log_metrics(metrics)
    mlflow.xgboost.log_model(model, "model")

2026/06/17 20:44:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



XGBoost (SMOTE)
  accuracy: 0.7779
 precision: 0.5768
    recall: 0.6123
        f1: 0.5940
   roc_auc: 0.8292

              precision    recall  f1-score   support

    Retained       0.86      0.84      0.85      1035
     Churned       0.58      0.61      0.59       374

    accuracy                           0.78      1409
   macro avg       0.72      0.72      0.72      1409
weighted avg       0.78      0.78      0.78      1409



In [49]:
with mlflow.start_run(run_name="xgboost_scale_pos_weight"):
    # Ratio of negative to positive class
    scale_pos_weight = (y_train_orig == 0).sum() / (y_train_orig == 1).sum()

    params = {
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 6,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "scale_pos_weight": scale_pos_weight,
        "random_state": 42,
        "eval_metric": "logloss"
    }
    model = xgb.XGBClassifier(**params)
    model.fit(X_train_orig, y_train_orig)  # note: original imbalanced data

    metrics, y_pred, y_proba = evaluate_model(model, X_test, y_test, "XGBoost (scale_pos_weight)")

    mlflow.log_params(params)
    mlflow.log_metrics(metrics)
    mlflow.log_param("imbalance_strategy", "scale_pos_weight")
    mlflow.xgboost.log_model(model, "model")

2026/06/17 20:44:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



XGBoost (scale_pos_weight)
  accuracy: 0.7630
 precision: 0.5394
    recall: 0.7326
        f1: 0.6213
   roc_auc: 0.8328

              precision    recall  f1-score   support

    Retained       0.89      0.77      0.83      1035
     Churned       0.54      0.73      0.62       374

    accuracy                           0.76      1409
   macro avg       0.71      0.75      0.72      1409
weighted avg       0.80      0.76      0.77      1409



In [50]:
with mlflow.start_run(run_name="lightgbm_smote"):
    params = {
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 6,
        "num_leaves": 31,
        "random_state": 42,
        "verbose": -1
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)

    metrics, y_pred, y_proba = evaluate_model(model, X_test, y_test, "LightGBM (SMOTE)")

    mlflow.log_params(params)
    mlflow.log_metrics(metrics)
    mlflow.lightgbm.log_model(model, "model")

2026/06/17 20:44:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/17 20:44:39 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



LightGBM (SMOTE)
  accuracy: 0.7871
 precision: 0.5934
    recall: 0.6283
        f1: 0.6104
   roc_auc: 0.8302

              precision    recall  f1-score   support

    Retained       0.86      0.84      0.85      1035
     Churned       0.59      0.63      0.61       374

    accuracy                           0.79      1409
   macro avg       0.73      0.74      0.73      1409
weighted avg       0.79      0.79      0.79      1409



In [51]:
runs = mlflow.search_runs(experiment_names=["churn-prediction"])
comparison = runs[['tags.mlflow.runName', 'metrics.roc_auc', 'metrics.f1',
                    'metrics.recall', 'metrics.precision', 'metrics.accuracy']]
comparison = comparison.sort_values('metrics.roc_auc', ascending=False)
print(comparison.to_string(index=False))

         tags.mlflow.runName  metrics.roc_auc  metrics.f1  metrics.recall  metrics.precision  metrics.accuracy
               random_forest         0.833696    0.608696        0.673797           0.555066          0.770050
logistic_regression_baseline         0.833209    0.611529        0.652406           0.575472          0.779986
    xgboost_scale_pos_weight         0.832824    0.621315        0.732620           0.539370          0.762952
              lightgbm_smote         0.830164    0.610390        0.628342           0.593434          0.787083
               xgboost_smote         0.829234    0.594034        0.612299           0.576826          0.777857


In [52]:
print("Active MLflow Database Path:", mlflow.get_tracking_uri())


Active MLflow Database Path: sqlite:///D:/Customer Churn Prediction/Customer-Churn-Prediction/notebooks/mlflow.db
